In [ ]:
# ! pip3 install tf-keras

In [1]:
from transformers import pipeline
import pandas as pd


/Users/negin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/negin/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_df = pd.read_csv('data/6/train.csv')

In [4]:
text = train_df['text'][0]
text

'Best sock ever made. Consistent quality...I highly recommend Gold Toe  socks.'

In [ ]:
# from transformers import pipeline

# pipe = pipeline("text-generation", model="Qwen/Qwen3-0.6B")

# def predict_sentiment(text):
#     prompt = f"""Classify sentiment.

# Return ONLY one word: positive or negative.

# Text: "{text}"
# Answer:"""

#     out = pipe(prompt, max_new_tokens=2, do_sample=False)[0]["generated_text"]
#     return out.replace(prompt, "").strip().lower()

# print(predict_sentiment(text))

In [7]:
def predict(model, prompt ,max_tokens=2):
    pipe = pipeline("text-generation", model=model)
    out = pipe(prompt, max_new_tokens=max_tokens, do_sample=False)[0]["generated_text"]
    return out.replace(prompt, "").strip().lower()

models = [
    "Qwen/Qwen3-0.6B",
    # "mistralai/Mistral-7B-Instruct-v0.2", # Too slow
    "Qwen/Qwen2.5-0.5B-Instruct",
    "microsoft/phi-2",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "distilgpt2"
]
prompt = f"""Classify sentiment.

    Return ONLY one word: positive or negative.

    Text: "{text}"
    Answer:"""

In [8]:
model = "Qwen/Qwen3-0.6B"
print(f"Model: {model}")
print(predict(model, prompt))

Model: Qwen/Qwen3-0.6B


Device set to use mps:0


positive


In [9]:
model = "Qwen/Qwen2.5-0.5B-Instruct"
print(f"Model: {model}")
print(predict(model, prompt))

Model: Qwen/Qwen2.5-0.5B-Instruct


Device set to use mps:0


positive


In [10]:
model = "microsoft/phi-2"
print(f"Model: {model}")
print(predict(model, prompt))

Model: microsoft/phi-2


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 28.18it/s]
Device set to use mps:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


!!


In [11]:
model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"Model: {model}")
print(predict(model, prompt))

Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0


Device set to use mps:0


positive


In [12]:
model = "distilgpt2"
print(f"Model: {model}")
print(predict(model, prompt))

Model: distilgpt2


Device set to use mps:0
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


"best


In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto")

text = "I really enjoyed this movie. The acting was great."

messages = [
    {"role": "system", "content": "You are a sentiment classifier. Reply with only one word: positive or negative."},
    {"role": "user", "content": f"Text: {text}"}
]

formatted = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=2,
    do_sample=False
)

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

`torch_dtype` is deprecated! Use `dtype` instead!


positive
